# Bias Detection & Fairness Analysis

This notebook implements bias detection and fairness analysis for the NovaCred credit application dataset.
As part of the Data Governance Task Force, the goal is to identify potential discrimination in historical
lending decisions and quantify bias using established fairness metrics.

**Analyses performed:**
1. Gender Bias — Disparate Impact Ratio (four-fifths rule)
2. Statistical Significance Testing — Chi-squared test
3. Fairlearn Fairness Metrics — Demographic parity difference
4. Rejection Reason Analysis — Gender-based rejection patterns
5. Proxy Discrimination — Spending categories and income as gender proxies
6. Intersectional Analysis — Approval patterns across income brackets and gender

## 1. Setup & Data Loading

We load the consolidated dataset produced by the Data Engineering pipeline. This dataset
merges the original application data with pivoted spending behavior into a single file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from fairlearn.metrics import (
    demographic_parity_difference,
    demographic_parity_ratio,
    MetricFrame,
    selection_rate
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

In [ ]:
# Load consolidated dataset (spending behavior merged into main applications)
df = pd.read_csv('../data/processed/cleaned_credit_applications.csv')

print(f'Dataset shape: {df.shape[0]} records x {df.shape[1]} columns')
print(f'\nColumns:')
for i, col in enumerate(df.columns):
    print(f'  {i+1:2d}. {col}')
print(f'\nGender distribution:')
print(df['applicant_info.gender'].value_counts(dropna=False))
print(f'\nOverall approval rate: {df["decision.loan_approved"].mean():.1%}')

In [ ]:
# Filter to Male/Female for binary fairness analysis
# 2 records with 'Unknown' gender are excluded
df_bias = df[df['applicant_info.gender'].isin(['Male', 'Female'])].copy()
df_bias['gender_binary'] = (df_bias['applicant_info.gender'] == 'Male').astype(int)
df_bias['approved_binary'] = df_bias['decision.loan_approved'].astype(int)

print(f'Records for bias analysis: {len(df_bias)} (excluded {len(df) - len(df_bias)} with Unknown gender)')
print(f'Female: {len(df_bias[df_bias["applicant_info.gender"]=="Female"])}')
print(f'Male:   {len(df_bias[df_bias["applicant_info.gender"]=="Male"])}')